# Marker Repo - submit lists

In [ ]:
%load_ext autoreload
%autoreload 2

## Loading packages

In [ ]:
import marker_repo as mr
import src.generate_metafile as gm
import src.validate_yaml as validate
import src.utils as utils

## Settings

In [ ]:
# The path where the lists of the Marker Repo are stored - probable 'REPO_PATH/lists'
REPO_LISTS_PATH = "/mnt/workspace/mkessle/projects/annotate_by_marker_and_features/lists"

# The path of the list to be added to the marker repo.
LIST_PATH = "/mnt/workspace/mkessle/projects/marker_repo/dbs/panglao_split/panglao_human_Blood"

# The path were the corresponding metadata file is located
# None, if you like to add the metadata manually
METADATA_PATH = None

## Get whitelists

Pull whitelist repository and update if necessary.

In [ ]:
mr.getWhitelists()

Get essential metadata: Organism and marker type

In [ ]:
ORGANISM = mr.select("organism")
MARKER_TYPE = mr.select("marker_type")

Read genes whitelist in order to filter and extend marker genes

In [ ]:
if MARKER_TYPE == "Genes":
    gene_dict = mr.get_gene_dict(ORGANISM)

## Read, filter, extend and transform marker list

Read marker list

In [ ]:
markers = mr.getList(LIST_PATH, "celltype")
display(markers)

Filter marker list

In [ ]:
# TODO - select filter: keep protein conding only? keep noncoding? keep ... ?
if MARKER_TYPE == "Genes":
    markers_filtered = markers[markers['Marker'].isin(gene_dict.keys())]
    display(markers_filtered)

Extend marker list

In [ ]:
if MARKER_TYPE == "Genes":
    markers_extended = mr.update_markers(markers_filtered, gene_dict)
    display(markers_extended)

Convert marker list in order to append it to the yaml file

In [ ]:
if MARKER_TYPE == "Genes":
    marker_dict = mr.dataframe_to_dict(markers_extended, info_col=0, marker_col=1)
else:
    # TODO - filter genomic regions?
    marker_dict = mr.dataframe_to_dict(markers, info_col=0, marker_col=1)
    
marker_list = []
for name in marker_dict.keys():
    marker_list.append({'name': name, 'markers': marker_dict[name]})

## Enter metadata

Enter general metadata, tags and add marker list(s) automatically.

In [ ]:
# TODO - solve problem of whitelist autocompletion (tissue)
gm.generate_file(REPO_LISTS_PATH, 2, False, marker_list, ORGANISM, MARKER_TYPE)

## Validation

In [ ]:
# TODO (?) markers are checked bevore adding
metadata = utils.read_in_yaml("lists/2.yaml")
validate.validate_file(metadata)